# GitHub → Colab / VS Code → Google Drive

代码来自 `https://github.com/BoldHu/CellQuant.git` 的 `master` 分支，运行目录为 `/content/CellQuant`。
在本地 VS Code 编辑并 push；云端停下作业后 pull，重启 kernel 再运行。
完整步骤见 [工作流说明](../docs/COLAB_VSCODE.md)。

VS Code 先选择 **Select Kernel → Colab**，再执行命令 **Colab: Mount Google Drive to Server...**。扩展会把挂载单元追加到 notebook 末尾，**先单独运行该单元并完成 Google 授权，再运行下方初始化流程**。
首次运行无需手动创建云盘中的 `CellQuant` 文件夹，挂载后会自动创建。`/content/drive/MyDrive` 是挂载入口；如果报它不存在，应先完成挂载，不能通过新建同名本地目录代替。
浏览器 Colab 可用下面显式的挂载开关。模型、数据、运行记录以版本化快照存到 Drive，实际计算读取本地盘。
默认不安装依赖、不下载模型、不运行真实实验、不备份大文件；本地离线执行只检查已有项目。

In [4]:
import os, sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/BoldHu/CellQuant.git"
BRANCH = "master"
IN_COLAB = Path("/content").is_dir()
PROJECT_ROOT = Path(os.environ.get("CELLQUANT_PROJECT_ROOT", "/content/CellQuant" if IN_COLAB else str(Path.cwd())))
# 可修改为自己的 Drive 项目目录；VS Code 先通过扩展命令挂载 Drive。
DRIVE_ROOT = os.environ.get("CELLQUANT_DRIVE_ROOT", "/content/drive/MyDrive/CellQuant") if IN_COLAB else None
DRIVE_ROOT = "/content/drive/MyDrive/CellQuant"
MOUNT_DRIVE_IN_BROWSER = False
INSTALL_BASE_DEPENDENCIES = True
RESTORE_AREAS = []  # 新 runtime 需要恢复时填 ["models", "datasets", "runs"]
RUN_G0 = True      # 首次验证时显式设 True，只运行 CPU 数学检查
BACKUP_AREAS = ["runs"]   # 完成阶段后填 ["runs"]；下载完成后另备份 models、datasets


In [5]:
PROJECT_ROOT = PROJECT_ROOT.resolve()
if IN_COLAB and not PROJECT_ROOT.exists():
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, str(PROJECT_ROOT)], check=True)
if not (PROJECT_ROOT / "pyproject.toml").is_file():
    raise FileNotFoundError("Set PROJECT_ROOT to the CellQuant checkout")
if IN_COLAB:
    origin = subprocess.check_output(["git", "-C", str(PROJECT_ROOT), "remote", "get-url", "origin"], text=True).strip()
    if origin.removesuffix(".git") != REPO_URL.removesuffix(".git"):
        raise RuntimeError("Existing checkout has a different origin; choose a new PROJECT_ROOT")
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT / "src"))
os.environ["PYTHONPATH"] = str(PROJECT_ROOT / "src") + (os.pathsep + os.environ["PYTHONPATH"] if os.environ.get("PYTHONPATH") else "")
os.environ["CELLQUANT_PROJECT_ROOT"] = str(PROJECT_ROOT)
os.environ["PYTHONUTF8"] = "1"
from cellquant.workspace import git_identity
print({"project": str(PROJECT_ROOT), "code": git_identity(PROJECT_ROOT)})


{'project': '/content/CellQuant', 'code': {'commit': 'b0425c2f92f983ee028c7a869a579e56e8e3b2e3', 'branch': 'master', 'dirty': False}}


In [6]:
if MOUNT_DRIVE_IN_BROWSER:
    if not IN_COLAB:
        raise RuntimeError("Drive mounting requires a Colab runtime")
    from google.colab import drive
    drive.mount("/content/drive")

from cellquant.workspace import configure_workspace
WORKSPACE_READY = False
if DRIVE_ROOT:
    record = configure_workspace(PROJECT_ROOT, DRIVE_ROOT)
    WORKSPACE_READY = True
    print(record)
else:
    print("Offline workspace check only; Drive persistence NOT_RUN")


RuntimeError: Mount Google Drive first; /content/drive/MyDrive is unavailable

In [ ]:
from cellquant.runtime import stream_command
from datetime import datetime, timezone
import uuid

def logged_command(argv, label, timeout=21600):
    stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S") + "_" + uuid.uuid4().hex[:8]
    # Backup logs live outside runs, so backup never copies its own changing log.
    log = PROJECT_ROOT / ".cache/workspace_logs" / f"{label}_{stamp}.log"
    result = stream_command(argv, log, cwd=PROJECT_ROOT, timeout_s=timeout)
    if result["returncode"]:
        raise RuntimeError(f"Command failed; see {log}")
    return result

if INSTALL_BASE_DEPENDENCIES:
    logged_command([sys.executable, "-m", "pip", "install", "-e", ".[dev]"], "install", 1800)
else:
    print("Dependency installation NOT_RUN; existing torch is preserved")
if RESTORE_AREAS:
    if not WORKSPACE_READY:
        raise RuntimeError("Configure mounted Drive first")
    logged_command([sys.executable, "-m", "cellquant.workspace", "restore", "--areas", *RESTORE_AREAS], "restore")


In [ ]:
if RUN_G0:
    from cellquant.runtime import atomic_json, environment_report, read_json
    output = PROJECT_ROOT / "runs/00_math"
    atomic_json(output / "environment.json", environment_report())
    for script, subdir in [("run_math_checks.py", "reference"), ("run_production_checks.py", "production")]:
        logged_command([sys.executable, f"scripts/{script}", "--output", str(output / subdir)], subdir, 600)
    print("Measured G0:", read_json(output / "production/G0.json")["status"])
else:
    print("G0 execution NOT_RUN in this notebook invocation")

if BACKUP_AREAS:
    if not WORKSPACE_READY:
        raise RuntimeError("Configure mounted Drive first")
    logged_command([sys.executable, "-m", "cellquant.workspace", "backup", "--areas", *BACKUP_AREAS], "backup")


## 后续使用

完成 G0 后打开 `01_Colab_Real_Layer_Pilot.ipynb`，并按 `docs/RUNBOOK.md` 准备固定 SpinQuant 和真实数据。
同一服务器上的不同 notebook 可能有不同 kernel；现有阶段 notebook 会读取本地 `.cache/colab_workspace.json` 恢复路径设置。

下载模型/数据后，在无下载或训练写入的阶段边界运行：
```python
logged_command([sys.executable, "-m", "cellquant.workspace", "backup", "--areas", "models", "datasets", "runs"], "backup")
```
Drive 中 `objects/` 是按 SHA-256 去重的文件，`snapshots/` 是版本索引，两者须一同保留。中断传输不发布成功索引；备份不改变科学门槛。
新的 runtime 先 clone、挂载、设置路径，再恢复需要的区域。更新代码后旧结果保留，恢复实验仍必须核对代码/环境/数据指纹。

停下所有云端作业后，打开官方扩展的 Colab terminal，运行：
```bash
cd /content/CellQuant
git status --short
git pull --ff-only origin master
python -m pip install -e '.[dev]'
```
有本地改动时先自行保存并解决；不要强制 reset。随后重启 notebook kernel，并重跑初始化单元。
如果没有 terminal，可在云端 notebook 用 `subprocess.run([...], check=True)` 执行同样的 Git 命令。
